# Run Biomni on Eval1 subset

This notebook runs the Biomni framework on the fixed Eval1 subset created in:

`project_subset.csv`

The goal is to generate Biomni outputs for the same benchmark queries used across the project.

Input:
`project_subset.csv`

Output:
`output_biomni.csv`

The output CSV is later scored with the Biomni Eval1 scorer using `pipelines/score_outputs.py`.


##### Imports

In [ ]:
from pathlib import Path
import json
import time
import pandas as pd
import os
from rich import print
from bio_agents.frameworks import FRAMEWORK_REGISTRY
from bio_agents.tasks.biomni_eval1.task import BiomniEval1Task

#### Auxiliary Functions

In [2]:
def safe_json(value):
    try:
        return json.dumps(value, ensure_ascii=False)
    except Exception:
        return str(value)

#### Load the fixed 30-query subset

In [ ]:
input_csv = Path("project_subset.csv")

df = pd.read_csv(input_csv)

print("Loaded rows:", len(df))
df.head()

Current working directory:

c:\Users\mbene\Documents\Bio_agents_Evaluation\data_subset

Files here:

[
    WindowsPath('create_eval1_subset.ipynb'),
    WindowsPath('output_edison.csv.csv'),
    WindowsPath('output_phylo.csv.csv'),
    WindowsPath('project_subset.csv'),
    WindowsPath('project_subset_10.csv'),
    WindowsPath('run_agents_on_subset.ipynb')
]

#### Configuration

In [ ]:
framework = "biomni"

# for now we use qwen3 for all frameworks, but this can be changed later
model = "qwen3"

output_csv = Path("output_biomni.csv")

commercial_mode = False
timeout_seconds = 600

print("Framework:", framework)
print("Model:", model)
print("Output:", output_csv)

## Agent runner

In [ ]:
runner = FRAMEWORK_REGISTRY[framework]()
results = []

print(f"[bold]Running {framework} on {len(df)} queries with model={model}[/bold]")

for idx, row in df.iterrows():
    print(f"\n========== Query {idx + 1}/{len(df)} ==========")
    print("Task:", row["task_name"])
    print("Instance:", row["task_instance_id"])

    task = BiomniEval1Task(
        task_name=row["task_name"],
        task_instance_id=row["task_instance_id"],
        prompt=row["input_query"],
    )

    task_input = task.get_input()
    tools = task.get_tools()

    start = time.perf_counter()

    try:
        agent_result = runner.run(
            task_input.prompt,
            tools,
            model,
            commercial_mode=commercial_mode,
            timeout_seconds=timeout_seconds,
        )

        duration_s = round(time.perf_counter() - start, 2)

        output_text = agent_result.output
        tool_calls = agent_result.tool_calls
        metadata = agent_result.metadata
        run_error = ""

        print("[green]Completed[/green]")
        print("Output:", output_text)

    except Exception as exc:
        duration_s = round(time.perf_counter() - start, 2)

        output_text = ""
        tool_calls = []
        metadata = {}
        run_error = str(exc)

        print("[red]ERROR[/red]", run_error)

    results.append(
        {
            "sample_index": row["sample_index"],
            "framework": framework,
            "model": model,
            "task_name": row["task_name"],
            "task_instance_id": row["task_instance_id"],
            "input_query": row["input_query"],
            "dataset_eval1_answer": row["dataset_eval1_answer"],
            "output": output_text,
            "duration_s": duration_s,
            "tool_calls_count": len(tool_calls),
            "tool_calls": safe_json(tool_calls),
            "metadata": safe_json(metadata),
            "run_error": run_error,
        }
    )

##### Save output

In [ ]:
results_df = pd.DataFrame(results)

results_df.to_csv(
    output_csv,
    index=False,
    encoding="utf-8-sig",
)

print("Saved to:", output_csv)
print("Rows:", len(results_df))

results_df.head()

In [11]:
pd.read_csv("output_biomni.csv").head()

,sample_index,framework,model,task_name,task_instance_id,input_query,dataset_eval1_answer,output,duration_s,tool_calls_count,tool_calls,metadata,run_error
0,1,biomni,qwen3,lab_bench_dbqa,387,The following is a multiple choice question ab...,A,The correct answer is determined by analyzing ...,4332.62,4,"[{""name"": ""step_0"", ""output"": ""===============...","{""model"": ""qwen3"", ""log_length"": 4, ""data_path...",NaN
1,2,biomni,qwen3,screen_gene_retrieval,259,Your task is to identify the gene with the str...,DHODH,The answer is <solution>DHODH</solution>,13138.43,2,"[{""name"": ""step_0"", ""output"": ""===============...","{""model"": ""qwen3"", ""log_length"": 2, ""data_path...",NaN
2,3,biomni,qwen3,gwas_variant_prioritization,88,Your task is to identify the most promising va...,rs2160860,1. [✓] Step 1 (corrected file path found) \n2...,5266.76,16,"[{""name"": ""step_0"", ""output"": ""===============...","{""model"": ""qwen3"", ""log_length"": 16, ""data_pat...",NaN
3,4,biomni,qwen3,gwas_causal_gene_opentargets,27,Your task is to identify likely causal genes w...,TBX4,<execute> \nThe user's instruction requires t...,5479.31,6,"[{""name"": ""step_0"", ""output"": ""===============...","{""model"": ""qwen3"", ""log_length"": 6, ""data_path...",NaN
4,5,biomni,qwen3,lab_bench_seqqa,187,The following is a multiple choice question ab...,D,ayout\n</think>\n\nI will now provide a respon...,2821.48,3,"[{""name"": ""step_0"", ""output"": ""===============...","{""model"": ""qwen3"", ""log_length"": 3, ""data_path...",NaN
